# 🔄 RS-LiDAR: Đồng Bộ & Chuyển Giao Dữ Liệu Sang Google Drive Cá Nhân Vĩnh Viễn

Notebook này phục vụ cho quy trình **sử dụng Google Drive Cá Nhân Vĩnh Viễn** để lưu trữ toàn bộ dữ liệu RS-LiDAR:
- **Lợi ích tuyệt đối**:
  - Dữ liệu checkpoints, latents và ảnh mẫu được lưu **vĩnh viễn** trên tài khoản cá nhân của bạn.
  - Mỗi tháng đổi tài khoản Colab mới, bạn **không cần tải về máy tính rồi upload lại** (tiết kiệm hàng chục GB băng thông và thời gian).
  - Chỉ cần **thêm lối tắt (Add shortcut)** là tài khoản Colab mới nhận diện toàn bộ tiến độ cũ ngay lập tức.

## 1. Gắn Kết Google Drive Của Tài Khoản Colab Hiện Tại

In [ ]:
import os, sys, shutil, glob, time
from google.colab import drive

# Gắn kết Google Drive an toàn
drive.mount('/content/drive')

base_drive = '/content/drive/My Drive' if os.path.exists('/content/drive/My Drive') else '/content/drive/MyDrive'
SRC_DIR = f'{base_drive}/RS-LiDAR'
DEST_DIR = f'{base_drive}/RS-LiDAR-Personal'

print('✅ Đã gắn kết Google Drive thành công!')
print('📁 Thư mục nguồn (Colab hiện tại):', SRC_DIR)
print('📁 Thư mục đích (Lối tắt Drive Cá Nhân):', DEST_DIR)

---
## 🌟 BƯỚC 2: CHUYỂN DỮ LIỆU TỪ COLAB HIỆN TẠI SANG DRIVE CÁ NHÂN

> **Yêu cầu trước khi chạy Cell dưới đây**:
> 1. Trên Google Drive cá nhân: Tạo thư mục `RS-LiDAR-Personal` -> Chia sẻ cho email Colab hiện tại (quyền **Editor**).
> 2. Trên Google Drive của Colab hiện tại: Vào mục **"Được chia sẻ với tôi" (Shared with me)** -> Chuột phải vào `RS-LiDAR-Personal` -> Chọn **Thêm lối tắt vào Drive (Add shortcut to Drive)** -> Lưu tại **Drive của tôi (My Drive)**.

In [ ]:
# @title 🚀 BẮT ĐẦU ĐỒNG BỘ TOÀN BỘ DỮ LIỆU SANG DRIVE CÁ NHÂN
assert os.path.exists(SRC_DIR), f"❌ Không tìm thấy thư mục nguồn {SRC_DIR}! Hãy chắc chắn tài khoản Colab này đã chạy thực nghiệm."
assert os.path.exists(DEST_DIR), f"❌ Không tìm thấy lối tắt {DEST_DIR}! Vui lòng làm theo hướng dẫn ở trên: Vào 'Shared with me' -> Thêm lối tắt 'RS-LiDAR-Personal' vào 'My Drive'."

print("=" * 70)
print("🔄 ĐANG BẮT ĐẦU SAO CHÉP DỮ LIỆU SANG DRIVE CÁ NHÂN VĨNH VIỄN...")
print("⚡ Quá trình diễn ra trực tiếp qua mạng nội bộ Google (siêu tốc, không tốn mạng nhà bạn).")
print("=" * 70)

start_time = time.time()
copied_files = 0
total_bytes = 0

folders_to_sync = ["Lookahead_samples", "Target_samples", "test_results"]

for folder_name in folders_to_sync:
    src_folder = os.path.join(SRC_DIR, folder_name)
    dest_folder = os.path.join(DEST_DIR, folder_name)
    
    if not os.path.exists(src_folder):
        continue
        
    print(f"\n📦 Đang đồng bộ thư mục: {folder_name}...")
    os.makedirs(dest_folder, exist_ok=True)
    
    for root, dirs, files in os.walk(src_folder):
        rel_path = os.path.relpath(root, src_folder)
        target_root = os.path.join(dest_folder, rel_path)
        os.makedirs(target_root, exist_ok=True)
        
        for file in files:
            src_file = os.path.join(root, file)
            dest_file = os.path.join(target_root, file)
            
            # Chỉ copy nếu file đích chưa có hoặc kích thước khác nhau
            if not os.path.exists(dest_file) or os.path.getsize(src_file) != os.path.getsize(dest_file):
                shutil.copy2(src_file, dest_file)
                copied_files += 1
                total_bytes += os.path.getsize(src_file)
                if copied_files % 50 == 0:
                    print(f"   • Đã sao chép {copied_files} files ({total_bytes / (1024**2):.1f} MB)...", end="\r")

elapsed = time.time() - start_time
print("\n" + "=" * 70)
print(f"🎉 ĐỒNG BỘ THÀNH CÔNG SANG DRIVE CÁ NHÂN!")
print(f"📊 Tổng số file đã sao chép: {copied_files} files")
print(f"💾 Tổng dung lượng: {total_bytes / (1024**3):.2f} GB")
print(f"⏱️ Thời gian thực hiện: {elapsed:.1f} giây ({total_bytes / (1024**2) / max(elapsed, 0.1):.1f} MB/s)")
print("📍 Dữ liệu hiện đã an toàn 100% trên tài khoản Google Drive cá nhân của bạn!")
print("=" * 70)

---
## 🌟 BƯỚC 3: KIỂM TRA TÍNH TOÀN VẸN TRÊN DRIVE CÁ NHÂN
Chạy cell này để kiểm tra xem trên Drive cá nhân đã đủ số lượng prompt và checkpoints chưa.

In [ ]:
# @title 🔍 KIỂM TRA TIẾN ĐỘ TRÊN DRIVE CÁ NHÂN
print("📊 BÁO CÁO DỮ LIỆU HIỆN CÓ TRÊN DRIVE CÁ NHÂN (RS-LiDAR-Personal):")
print("-" * 70)

# 1. Lookahead samples
look_folders = sorted(glob.glob(f"{DEST_DIR}/Lookahead_samples/*"))
if look_folders:
    print(f"📁 [Lookahead_samples] Tìm thấy {len(look_folders)} thư mục:")
    for lf in look_folders:
        p_count = len(glob.glob(f"{lf}/[0-9]*/results.json"))
        lat_count = len(glob.glob(f"{lf}/[0-9]*/samples/latent.pt"))
        print(f"   • {os.path.basename(lf)}: {p_count}/553 prompts đã chấm reward | {lat_count}/553 latents có sẵn")
else:
    print("ℹ️ Chưa có thư mục Lookahead_samples.")

# 2. Target samples
targ_folders = sorted(glob.glob(f"{DEST_DIR}/Target_samples/*"))
if targ_folders:
    print(f"\n📁 [Target_samples] Tìm thấy {len(targ_folders)} thí nghiệm đã sinh ảnh:")
    for tf in targ_folders:
        p_count = len(glob.glob(f"{tf}/[0-9]*/results.json"))
        has_geneval = os.path.exists(f"{tf}/geneval_summary.csv")
        has_pub = os.path.exists(f"{tf}/table2_publication_summary.csv")
        status_str = []
        if has_geneval: status_str.append("GenEval ✅")
        if has_pub: status_str.append("Table 2 ✅")
        extra = f" ({', '.join(status_str)})" if status_str else ""
        print(f"   • {os.path.basename(tf)}: {p_count}/553 prompts hoàn thành{extra}")
else:
    print("ℹ️ Chưa có thư mục Target_samples.")

print("-" * 70)
print("✅ MỌI THỨ ĐÃ SẴN SÀNG ĐỂ BẠN ĐỔI SANG TÀI KHOẢN COLAB MỚI BẤT KỲ LÚC NÀO!")

---
## 💡 HƯỚNG DẪN DÀNH CHO TÀI KHOẢN COLAB MỚI (THÁNG SAU)

Khi bạn đăng nhập vào **tài khoản Colab mới** vào tháng tiếp theo:

1. **Trên Drive Cá Nhân**:
   - Chuột phải vào `RS-LiDAR-Personal` -> Chọn **Chia sẻ** -> Nhập email Colab mới (quyền **Editor**).
2. **Trên Drive của tài khoản Colab Mới**:
   - Vào mục **"Được chia sẻ với tôi" (Shared with me)**.
   - Chuột phải vào `RS-LiDAR-Personal` -> Chọn **Thêm lối tắt vào Drive (Add shortcut to Drive)**.
   - Đổi tên lối tắt thành **`RS-LiDAR`** và đặt ngay tại **Drive của tôi (My Drive)**.
3. **Mở Colab mới và chạy**:
   - Mở `LiDAR_Table2_Replication_Colab.ipynb`.
   - Chạy bình thường: Code sẽ đọc/ghi thẳng vào Drive cá nhân của bạn, nhận diện toàn bộ kết quả cũ và tự động skip các prompt đã xong!